# ⚙️ SEMAINE 6 : Optimisation et Validation

**Projet:** Détection précoce DT1 - Cameroun  
**Objectifs:**
- Optimiser les hyperparamètres (GridSearchCV, RandomizedSearchCV)
- Validation croisée (K-Fold, Stratified K-Fold)
- Sélection du modèle final
- Sauvegarde du modèle

**Durée estimée:** 12-14 heures

---

In [ ]:
# Import des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     RandomizedSearchCV, cross_val_score, 
                                     StratifiedKFold)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, make_scorer)
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliothèques importées!")

In [ ]:
# Charger les données
df = pd.read_csv('../2_DONNEES/processed/dataset_clean.csv')
features = ['age', 'IMC', 'glycemie_jeun', 'HbA1c', 'ANP32A_IT1', 'ESCO2', 'NBPF1']
X = df[features]
y = df['diagnostic']

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📂 Dataset prêt: {len(X_train)} train / {len(X_test)} test")

## 🔄 Partie 1 : Validation Croisée (Cross-Validation)

La validation croisée permet d'évaluer la robustesse d'un modèle en le testant sur plusieurs sous-ensembles des données.

In [ ]:
print("🔄 VALIDATION CROISÉE\n")

# Créer un modèle Random Forest de base
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# K-Fold stratifié (préserve la proportion DT1/Sain)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Définir plusieurs métriques
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Évaluer chaque métrique avec validation croisée
results_cv = {}
for metric_name, metric in scoring.items():
    scores = cross_val_score(rf, X_train, y_train, cv=skf, scoring=metric, n_jobs=-1)
    results_cv[metric_name] = scores
    print(f"{metric_name.upper():12} : {scores.mean():.3f} ± {scores.std():.3f}")
    print(f"             Scores par fold: {scores.round(3)}\n")

print("💡 Un faible écart-type indique que le modèle est stable")

In [ ]:
# Visualisation des scores de validation croisée
fig, ax = plt.subplots(figsize=(10, 6))

metrics_names = list(results_cv.keys())
means = [results_cv[m].mean() for m in metrics_names]
stds = [results_cv[m].std() for m in metrics_names]

x = np.arange(len(metrics_names))
ax.bar(x, means, yerr=stds, capsize=5, alpha=0.7, edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metrics_names])
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Validation Croisée (5-Fold) - Random Forest', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Ajouter les valeurs au-dessus des barres
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.02, f'{m:.3f}\n±{s:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 🔍 Partie 2 : Optimisation avec GridSearchCV

GridSearchCV teste **toutes** les combinaisons d'hyperparamètres dans une grille définie.

In [ ]:
print("🔍 GRIDSEARCHCV - Random Forest\n")

# Définir la grille d'hyperparamètres
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

print(f"Nombre de combinaisons à tester: {np.prod([len(v) for v in param_grid_rf.values()])}")
print("⏳ Cela peut prendre plusieurs minutes...\n")

# Créer le GridSearchCV avec Recall comme métrique principale
grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='recall',  # Métrique prioritaire
    n_jobs=-1,
    verbose=1
)

# Entraîner
grid_search_rf.fit(X_train, y_train)

print(f"\n✅ Recherche terminée!")
print(f"\n🏆 Meilleurs hyperparamètres:")
for param, value in grid_search_rf.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Meilleur score (Recall CV): {grid_search_rf.best_score_:.3f}")

In [ ]:
# Évaluer le meilleur modèle sur le test set
best_rf = grid_search_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

print("📈 PERFORMANCE SUR TEST SET\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best_rf)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred_best_rf)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred_best_rf)*100:.2f}%  ⭐")
print(f"F1-score:  {f1_score(y_test, y_pred_best_rf):.3f}")

## 🎲 Partie 3 : RandomizedSearchCV (Alternative plus rapide)

RandomizedSearchCV teste un **échantillon aléatoire** de combinaisons, plus rapide que GridSearch.

In [ ]:
print("🎲 RANDOMIZEDSEARCHCV - XGBoost\n")

# Calculer scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Définir la distribution d'hyperparamètres
param_dist_xgb = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'scale_pos_weight': [1, scale_pos_weight, scale_pos_weight * 1.5]
}

print(f"Nombre total de combinaisons possibles: {np.prod([len(v) for v in param_dist_xgb.values()])}")
print(f"On va en tester 50 aléatoirement\n")

# Créer le RandomizedSearchCV
random_search_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_dist_xgb,
    n_iter=50,  # Nombre de combinaisons à tester
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='recall',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Entraîner
random_search_xgb.fit(X_train, y_train)

print(f"\n✅ Recherche terminée!")
print(f"\n🏆 Meilleurs hyperparamètres:")
for param, value in random_search_xgb.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Meilleur score (Recall CV): {random_search_xgb.best_score_:.3f}")

In [ ]:
# Évaluer le meilleur XGBoost sur test set
best_xgb = random_search_xgb.best_estimator_
y_pred_best_xgb = best_xgb.predict(X_test)

print("📈 PERFORMANCE SUR TEST SET\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best_xgb)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred_best_xgb)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred_best_xgb)*100:.2f}%  ⭐")
print(f"F1-score:  {f1_score(y_test, y_pred_best_xgb):.3f}")

## 🏆 Partie 4 : Sélection du Modèle Final

In [ ]:
# Comparer les deux modèles optimisés
resultats_finaux = pd.DataFrame({
    'Modèle': ['Random Forest (GridSearch)', 'XGBoost (RandomSearch)'],
    'Recall CV': [grid_search_rf.best_score_, random_search_xgb.best_score_],
    'Accuracy Test': [
        accuracy_score(y_test, y_pred_best_rf),
        accuracy_score(y_test, y_pred_best_xgb)
    ],
    'Recall Test': [
        recall_score(y_test, y_pred_best_rf),
        recall_score(y_test, y_pred_best_xgb)
    ],
    'Precision Test': [
        precision_score(y_test, y_pred_best_rf),
        precision_score(y_test, y_pred_best_xgb)
    ],
    'F1 Test': [
        f1_score(y_test, y_pred_best_rf),
        f1_score(y_test, y_pred_best_xgb)
    ]
})

print("🏆 COMPARAISON DES MODÈLES OPTIMISÉS\n")
print(resultats_finaux.round(3))

# Sélectionner le meilleur selon Recall Test
best_idx = resultats_finaux['Recall Test'].idxmax()
modele_final_nom = resultats_finaux.loc[best_idx, 'Modèle']
modele_final = best_xgb if 'XGBoost' in modele_final_nom else best_rf

print(f"\n⭐ MODÈLE FINAL SÉLECTIONNÉ: {modele_final_nom}")
print(f"   Recall Test: {resultats_finaux.loc[best_idx, 'Recall Test']*100:.2f}%")

## 💾 Partie 5 : Sauvegarde du Modèle Final

In [ ]:
# Créer le dossier models s'il n'existe pas
import os
os.makedirs('../4_MODELES/models', exist_ok=True)

# Sauvegarder le modèle final
model_path = '../4_MODELES/models/final_model.pkl'
joblib.dump(modele_final, model_path)

print(f"💾 Modèle sauvegardé: {model_path}")

# Sauvegarder aussi les features utilisées
features_path = '../4_MODELES/models/features.pkl'
joblib.dump(features, features_path)
print(f"💾 Features sauvegardées: {features_path}")

# Tester le chargement
print("\n🔄 Test de chargement...")
modele_charge = joblib.load(model_path)
features_chargees = joblib.load(features_path)

# Vérifier qu'il fonctionne
y_pred_test_load = modele_charge.predict(X_test)
recall_test_load = recall_score(y_test, y_pred_test_load)

print(f"✅ Modèle chargé avec succès!")
print(f"   Features: {features_chargees}")
print(f"   Recall (vérification): {recall_test_load*100:.2f}%")

## 📝 Partie 6 : Rapport Final du Modèle

In [ ]:
# Créer un rapport détaillé
rapport = f"""
{'='*70}
RAPPORT FINAL - MODÈLE DE DÉTECTION DT1 CAMEROUN
{'='*70}

📊 MODÈLE SÉLECTIONNÉ: {modele_final_nom}

🔧 HYPERPARAMÈTRES OPTIMAUX:
"""

if 'XGBoost' in modele_final_nom:
    for param, value in random_search_xgb.best_params_.items():
        rapport += f"   - {param}: {value}\n"
else:
    for param, value in grid_search_rf.best_params_.items():
        rapport += f"   - {param}: {value}\n"

y_pred_final = modele_final.predict(X_test)
rapport += f"""
📈 PERFORMANCES SUR TEST SET (n={len(X_test)}):
   - Accuracy:  {accuracy_score(y_test, y_pred_final)*100:.2f}%
   - Precision: {precision_score(y_test, y_pred_final)*100:.2f}%
   - Recall:    {recall_score(y_test, y_pred_final)*100:.2f}%  ⭐ (PRIORITAIRE)
   - F1-score:  {f1_score(y_test, y_pred_final):.3f}

🎯 INTERPRÉTATION CLINIQUE:
   - Sur 100 patients DT1, le modèle en détecte {recall_score(y_test, y_pred_final)*100:.0f}
   - Sur 100 patients prédits DT1, {precision_score(y_test, y_pred_final)*100:.0f} le sont réellement

💾 FICHIERS SAUVEGARDÉS:
   - Modèle: {model_path}
   - Features: {features_path}

📅 Date: Novembre 2025
{'='*70}
"""

print(rapport)

# Sauvegarder le rapport
with open('../4_MODELES/models/rapport_modele_final.txt', 'w', encoding='utf-8') as f:
    f.write(rapport)

print("\n💾 Rapport sauvegardé: ../4_MODELES/models/rapport_modele_final.txt")

## 🎯 Conclusion

### Points clés:

✅ **Validation croisée** évalue la robustesse du modèle  
✅ **GridSearchCV** optimise les hyperparamètres (exhaustif mais lent)  
✅ **RandomizedSearchCV** alternative plus rapide  
✅ **Recall prioritaire** pour minimiser les faux négatifs en médecine  
✅ **Sauvegarde avec joblib** pour réutiliser le modèle  

### Prochaines étapes:

1. Interpréter le modèle (SHAP, LIME)
2. Analyser les erreurs de prédiction
3. Intégrer dans l'application Streamlit

### 📅 Prochaine étape:
**Semaine 7:** Interprétabilité et explications des prédictions (SHAP, LIME)

---

*Notebook créé pour le projet DT1 Cameroun - Master 2 Biophysique - 2025*